# 03 Transfer Learning with CNNs

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Load a **pre-trained** CNN (e.g. MobileNetV2) and reuse its feature layers
- **Freeze** the base and train only a new head on a small dataset (e.g. MNIST or a subset)
- See why we use transfer learning instead of training from scratch when data is limited

---

## 🌍 Real life

**Where is this used?** Transfer learning is used in **medical imaging**, **custom classifiers** (e.g. product recognition), and **mobile vision** when we have limited labeled data.

**In this notebook we use** a **pre-trained model** (e.g. MobileNetV2) and **freeze** its base, then **train only the new head** on our data. We use **transfer learning** (instead of training from scratch) **because** the pre-trained layers already learned useful features (edges, textures); we reuse them and need **less data and time**.

**📌 Covers slide(s):** **20** — Transfer Learning (VGG, ResNet, fine-tuning). *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell below. First run may download the pre-trained weights.

## Theory (short)

- **Transfer learning:** Take a model trained on a large dataset (e.g. ImageNet); **freeze** most layers and **replace the head** (classification layer) for our classes.
- **Freeze vs fine-tune:** Freeze = don't update base weights; only train the new head. Fine-tune = later unfreeze some layers and train with a small learning rate.
- **When to use:** Use when we have **limited data** or **similar domain** (e.g. natural images). Pre-trained features generalize well.
- **We use a pre-trained base** instead of training from scratch so we reuse learned features and train faster with less data.

## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow/Keras (MobileNetV2 or similar), NumPy. We use MNIST resized to the model input size (e.g. 96×96 or 224×224) so we don't need a new dataset.

**Dataset:** Real — MNIST (resized for transfer learning).

**Outputs:** Model summary, training loss/accuracy for the new head (2 epochs), and test accuracy.

## Step 1: Imports and load pre-trained base (we use MobileNetV2 so it runs quickly; in production you might use ResNet)

In [1]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

if HAS_TF:
    base = keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights="imagenet")
    base.trainable = False
    print("Base (frozen) trainable params:", sum(np.prod(v.shape) for v in base.trainable_variables))
else:
    print("Install TensorFlow: pip install tensorflow")

Base (frozen) trainable params: 0


## Step 2: Build model = base + new head (we use transfer learning instead of training from scratch to reuse features)

In [2]:
if HAS_TF:
    x = base.output
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(10, activation="softmax")(x)
    model = keras.Model(base.input, x)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("Model built. Only the new Dense layer is trainable.")

Model built. Only the new Dense layer is trainable.


## Step 3: Prepare MNIST as 96×96 RGB (to match MobileNetV2 input)

In [3]:
if HAS_TF:
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    x_train = tf.image.resize(x_train[..., np.newaxis], (96, 96))
    x_test = tf.image.resize(x_test[..., np.newaxis], (96, 96))
    x_train = tf.repeat(x_train, 3, axis=-1).numpy().astype(np.float32) / 255.0
    x_test = tf.repeat(x_test, 3, axis=-1).numpy().astype(np.float32) / 255.0
    x_train_small = x_train[:5000]
    y_train_small = y_train[:5000]
    print("Train subset:", x_train_small.shape)

Train subset: (5000, 96, 96, 3)


## Step 4: Train only the new head (2 epochs)

In [4]:
if HAS_TF:
    history = model.fit(x_train_small, y_train_small, validation_data=(x_test, y_test), epochs=2, batch_size=64, verbose=1)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Test accuracy: %.4f" % acc)

Epoch 1/2


 1/79 [..............................] - ETA: 44s - loss: 2.7231 - accuracy: 0.1250

 3/79 [>.............................] - ETA: 3s - loss: 2.5808 - accuracy: 0.1094 

 5/79 [>.............................] - ETA: 3s - loss: 2.3927 - accuracy: 0.1719

 7/79 [=>............................] - ETA: 3s - loss: 2.2578 - accuracy: 0.2121

...

75/79 [===========================>..] - ETA: 0s - loss: 0.2777 - accuracy: 0.9287

77/79 [============================>.] - ETA: 0s - loss: 0.2758 - accuracy: 0.9292

79/79 [==============================] - ETA: 0s - loss: 0.2752 - accuracy: 0.9294

79/79 [==============================] - 11s 138ms/step - loss: 0.2752 - accuracy: 0.9294 - val_loss: 0.2467 - val_accuracy: 0.9328


Test accuracy: 0.9328


## 🧩 Mini-exercise

**Try it:** Unfreeze the last few layers of the base (e.g. set `base.trainable = True` and recompile), then train for 1 more epoch with a small learning rate (e.g. 1e-5). Does accuracy improve?

---

## ✅ Summary

**What you did:** Loaded a pre-trained base (MobileNetV2), froze it, added a new head, and trained only the head on MNIST (resized to 96×96 RGB).

**In real life you'd also:** Use your own dataset, optionally fine-tune the last few layers, and tune learning rate.

**The main idea:** Transfer learning reuses pre-trained features so we need less data and time; freeze the base and train the new head first.

**Next:** `06_pretrained_cnn_architectures` explores ResNet/VGG/Inception; `07_training_cnn_image_datasets` covers full training pipelines.